<a href="https://colab.research.google.com/github/Cheetah-lhp/MachineLearning/blob/main/Group15_introDL_CatBoost_MIMIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pickle

trainDataset = pickle.load(open('train.pkl', 'rb'))
testDataset = pickle.load(open('test.pkl', 'rb'))

In [9]:
import numpy as np
import pandas as pd


def extract_feature(values):
    if len(values) == 0:
        return {
            'mean': np.nan,
            'max': np.nan,
            'min': np.nan,
            'std': np.nan,
            'last': np.nan
        }

    vals = [v['value'] for v in values if v['value'] is not None]

    if len(vals) == 0:
        return {
            'mean': np.nan,
            'max': np.nan,
            'min': np.nan,
            'std': np.nan,
            'last': np.nan
        }

    return {
        'mean': np.mean(vals),
        'max': np.max(vals),
        'min': np.min(vals),
        'std': np.std(vals),
        'last': vals[-1]
    }

def process_dataset(data):
    rows = []

    for pid, patient in data.items():
        row = {}

        # ===== numeric time-series =====
        time_series_features = [
            'pao2', 'paco2', 'spo2',
            'pao2fio2ratio', 'totalco2',
            'baseexcess', 'ph', 'lactatebg'
        ]

        for feat in time_series_features:
            stats = extract_feature(patient.get(feat, []))
            for k, v in stats.items():
                row[f"{feat}_{k}"] = v

        # ===== categorical =====
        specimen = patient.get('specimen', [])
        if len(specimen) > 0:
            row['specimen_last'] = specimen[-1]['value']
        else:
            row['specimen_last'] = 'unknown'

        # ===== static =====
        row['age'] = patient.get('age_at_admission', np.nan)

        # ===== target =====
        row['target'] = patient.get('target', 0)

        rows.append(row)

    return pd.DataFrame(rows)



In [10]:
df = process_dataset(trainDataset)

In [11]:
print(df.head())

    pao2_mean  pao2_max  pao2_min    pao2_std  pao2_last  paco2_mean  \
0  110.333333     173.0      60.0   46.949145       60.0   25.333333   
1  238.666667     394.0      65.0  134.937846       65.0   45.666667   
2  141.857143     241.0      68.0   73.197245       68.0   43.142857   
3   85.000000     101.0      69.0   16.000000       69.0   33.000000   
4  233.363636     387.0      96.0  107.819945       97.0   37.727273   

   paco2_max  paco2_min  paco2_std  paco2_last  ...    ph_std  ph_last  \
0       26.0       24.0   0.942809        26.0  ...  0.032016     7.39   
1       54.0       31.0  10.402991        52.0  ...  0.111455     7.31   
2       62.0       33.0   8.424939        33.0  ...  0.078272     7.40   
3       35.0       31.0   2.000000        31.0  ...  0.080416     7.52   
4       44.0       32.0   3.620420        39.0  ...  0.045016     7.35   

   lactatebg_mean  lactatebg_max  lactatebg_min  lactatebg_std  \
0        1.200000            1.4            1.0       0.

In [12]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [13]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

X = df.drop(columns=['target'])
y = df['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cat_features = ['specimen_last']

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    early_stopping_rounds=50,
    eval_metric='AUC',
    loss_function='Logloss',
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)

0:	test: 0.6999639	best: 0.6999639 (0)	total: 89.9ms	remaining: 1m 29s
100:	test: 0.7851746	best: 0.7851746 (100)	total: 3.9s	remaining: 34.7s
200:	test: 0.7889444	best: 0.7894246 (190)	total: 11s	remaining: 43.9s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7894246131
bestIteration = 190

Shrink model to first 191 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', verbose=100)

In [14]:
test_df = process_dataset(testDataset)

In [15]:
test_df

,pao2_mean,pao2_max,pao2_min,pao2_std,pao2_last,paco2_mean,paco2_max,paco2_min,paco2_std,paco2_last,...,ph_std,ph_last,lactatebg_mean,lactatebg_max,lactatebg_min,lactatebg_std,lactatebg_last,specimen_last,age,target
0,181.500000,225.0,138.0,43.500000,138.0,38.500000,41.0,36.0,2.500000,41.0,...,0.015000,7.43,1.550000,1.7,1.4,0.150000,1.4,ART.,77,0
1,115.000000,202.0,75.0,48.520982,75.0,53.714286,92.0,40.0,16.219162,54.0,...,0.098768,7.35,1.225000,1.6,0.9,0.286138,0.9,ART.,82,0
2,132.333333,215.0,66.0,61.915713,116.0,66.333333,72.0,59.0,5.436502,59.0,...,0.059722,7.41,1.033333,1.2,0.8,0.169967,1.1,ART.,84,0
3,122.857143,172.0,92.0,25.985867,126.0,37.714286,50.0,25.0,8.012745,42.0,...,0.065652,7.44,1.583333,2.1,1.1,0.357849,1.1,ART.,70,0
4,128.750000,254.0,81.0,72.627044,82.0,41.250000,45.0,40.0,2.165064,40.0,...,0.025860,7.45,1.400000,1.5,1.3,0.081650,1.3,ART.,72,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2038,88.500000,100.0,77.0,11.500000,77.0,36.000000,39.0,33.0,3.000000,33.0,...,0.025000,7.31,1.300000,1.3,1.3,0.000000,1.3,ART.,76,0
2039,160.300000,253.0,63.0,55.553668,179.0,65.000000,92.0,45.0,14.587666,45.0,...,0.026722,7.12,4.537500,8.2,0.8,2.651385,8.2,ART.,64,0
2040,201.000000,201.0,201.0,0.000000,201.0,72.000000,72.0,72.0,0.000000,72.0,...,0.050000,7.16,6.300000,6.3,6.3,0.000000,6.3,ART.,82,0
2041,298.500000,430.0,171.0,99.426606,171.0,37.300000,48.0,30.0,5.950630,38.0,...,0.048415,7.40,1.625000,2.7,1.2,0.625999,1.4,ART.,63,0


In [16]:
from sklearn.metrics import roc_auc_score

X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

X_test = X_test[X_train.columns]

X_test['specimen_last'] = X_test['specimen_last'].astype(str)

y_pred_proba = model.predict_proba(X_test)[:, 1]

In [17]:
import pickle

testDataset = "test.pkl"
with open(testDataset, 'rb') as f:
    dataTest = pickle.load(f)

predFile = "group15.csv"

with open(predFile, "w") as f:
    f.write("id,probability,prediction\n")

    for patient_id, patient_data in dataTest.items():
        row_df = process_dataset({patient_id: patient_data})
        X_sample = row_df.drop(columns=['target'], errors='ignore')
        for col in X_train.columns:
            if col not in X_sample.columns:
                X_sample[col] = -999

        X_sample = X_sample[X_train.columns]
        X_sample['specimen_last'] = X_sample['specimen_last'].astype(str)
        prob = model.predict_proba(X_sample)[0, 1]
        pred = int(prob >= 0.5)

        f.write(f"{patient_id},{prob:.4f},{pred}\n")